# Générateurs

## Écrire un itérateur

Un itérateur garde l'état du parcours dans ses attributs : `__iter__` le renvoie lui-même, et `__next__` donne l'élément suivant, ou lève `StopIteration` quand il n'y en a plus.

1. Écrivez la classe `Fibonacci(limite)`, un itérateur sur les nombres de Fibonacci strictement inférieurs à `limite` : `list(Fibonacci(20))` donne `[0, 1, 1, 2, 3, 5, 8, 13]`.
2. Créez `suite = Fibonacci(20)` et affichez `list(suite)` deux fois. Pourquoi la seconde liste est-elle vide ?
3. Écrivez la classe `Journal(lignes)`, dont la méthode `__iter__` est un générateur qui produit les lignes non vides. Avec `journal = Journal(["a", "", "b"])`, affichez `list(journal)` deux fois : les deux listes doivent valoir `["a", "b"]`. Pourquoi le second parcours marche-t-il cette fois ?

In [ ]:
# Votre code ici
from collections.abc import Iterator


class Fibonacci:
  def __init__(self, limite: int) -> None: ...

  def __iter__(self) -> "Fibonacci": ...

  def __next__(self) -> int: ...


class Journal:
  def __init__(self, lignes: list[str]) -> None: ...

  def __iter__(self) -> Iterator[str]: ...

### Conseils supplémentaires

`__init__` range la limite et l'état du parcours dans des attributs : `self.courant = 0`, `self.suivant = 1`. `__iter__` renvoie `self`. `__next__` lève `StopIteration` quand `self.courant` atteint la limite ; sinon, il garde la valeur à renvoyer, avance les deux attributs (le nouveau `suivant` est la somme des deux anciens), et renvoie cette valeur. Dans `Journal`, `__iter__` contient un `yield` : chaque parcours l'appelle, et obtient un générateur neuf.

```python
class Fibonacci:
  def __init__(self, limite: int) -> None:
    self.limite = limite
    self.courant = 0
    self.suivant = 1

  def __iter__(self) -> "Fibonacci":
    return self

  def __next__(self) -> int:
    if self.courant >= self.limite:
      raise ...
    valeur = self.courant
    self.courant, self.suivant = ...
    return valeur


class Journal:
  def __init__(self, lignes: list[str]) -> None:
    self.lignes = lignes

  def __iter__(self) -> Iterator[str]:
    for ligne in self.lignes:
      if ligne:
        yield ...
```

### Solution

In [ ]:
from collections.abc import Iterator


class Fibonacci:
  def __init__(self, limite: int) -> None:
    self.limite = limite
    self.courant = 0
    self.suivant = 1

  def __iter__(self) -> "Fibonacci":
    return self

  def __next__(self) -> int:
    if self.courant >= self.limite:
      raise StopIteration
    valeur = self.courant
    self.courant, self.suivant = self.suivant, self.courant + self.suivant
    return valeur


class Journal:
  def __init__(self, lignes: list[str]) -> None:
    self.lignes = lignes

  def __iter__(self) -> Iterator[str]:
    for ligne in self.lignes:
      if ligne:
        yield ligne


suite = Fibonacci(20)
print(list(suite))  # [0, 1, 1, 2, 3, 5, 8, 13]
print(list(suite))  # []

journal = Journal(["a", "", "b"])
print(list(journal))  # ['a', 'b']
print(list(journal))  # ['a', 'b']

`Fibonacci` est un itérateur : `__iter__` renvoie `self`, et le second `list` repart d'un état épuisé. `Journal` est un itérable : chaque appel de `__iter__` crée un nouveau générateur, avec son propre état, et l'objet ne garde que les données.

## Générateur de dates infini

Écrivez la fonction `genere_dates` qui, à partir d'une date de départ comprise, génère toutes les dates successives séparées d'une semaine. Pour un départ le 1900-01-01, les trois premières dates sont 1900-01-01, 1900-01-08 et 1900-01-15.

**Documentation utile**

- [`datetime.timedelta`](https://docs.python.org/fr/3/library/datetime.html#datetime.timedelta)

In [ ]:
# Votre code ici
import datetime
from collections.abc import Iterator


def genere_dates(date: datetime.date) -> Iterator[datetime.date]: ...

Affichez les 10 premières dates avec une boucle `for`, que vous arrêtez avec `break`.

Écrivez ensuite `prend(iterable, n)`, un générateur qui produit les `n` premiers éléments d'un itérable sans en lire un de plus : avec `nombres = iter(range(10))`, après `list(prend(nombres, 3))`, `next(nombres)` doit donner 3. Affichez les 10 mêmes dates avec `prend`, puis avec `itertools.islice`.

In [ ]:
# Votre code ici
def prend(iterable, n): ...

### Conseils supplémentaires

`datetime.timedelta(days=7)` représente une semaine, et s'ajoute à une date. Dans `genere_dates`, produisez la date courante avant de l'avancer : dans l'ordre inverse, la date de départ n'est jamais produite. Pour `prend`, `zip` s'arrête dès que l'un de ses arguments est épuisé, et il les lit dans l'ordre : `zip(range(n), iterable)` voit `range` s'arrêter avant de toucher à l'itérable, alors que `zip(iterable, range(n))` lit un élément de trop. Attendu : 10 dates, de 1900-01-01 à 1900-03-05.

```python
def genere_dates(date: datetime.date) -> Iterator[datetime.date]:
  semaine = datetime.timedelta(days=7)
  while True:
    yield ...
    date += ...


def prend(iterable, n):
  for _, element in zip(range(n), iterable):
    yield ...


for date in prend(genere_dates(datetime.date(1900, 1, 1)), 10):
  print(date)

for date in itertools.islice(..., 10):
  print(date)
```

### Solution

In [ ]:
import datetime
import itertools
from collections.abc import Iterable, Iterator


def genere_dates(date: datetime.date) -> Iterator[datetime.date]:
  semaine = datetime.timedelta(days=7)
  while True:
    yield date
    date += semaine


for i, d in enumerate(genere_dates(datetime.date(1900, 1, 1))):
  if i == 10:
    break
  print(d)


def prend(iterable: Iterable, n: int) -> Iterator:
  for _, element in zip(range(n), iterable):
    yield element


for date in prend(genere_dates(datetime.date(1900, 1, 1)), 10):
  print(date)

for date in itertools.islice(genere_dates(datetime.date(1900, 1, 1)), 10):
  print(date)

nombres = iter(range(10))
print(list(prend(nombres, 3)), next(nombres))  # [0, 1, 2] 3

## Pipeline paresseux sur un gros fichier

Les générateurs enchaînent des transformations sans jamais stocker les données intermédiaires. La cellule suivante crée un journal de 500 000 requêtes (`numéro;utilisateur;durée_ms`) et fournit `pic_memoire`, qui mesure le pic de mémoire d'un appel.

1. Écrivez trois générateurs : `lit_lignes(chemin)` qui produit les lignes du fichier, `analyse(lignes)` qui produit des tuples `(utilisateur, durée)`, et `lentes(requetes, seuil)` qui ne garde que les requêtes plus longues que `seuil`.
2. Écrivez `total_lentes_generateurs(chemin)`, qui enchaîne vos générateurs et renvoie avec `sum` la durée totale des requêtes de plus de 900 ms.
3. Écrivez `total_lentes_listes(chemin)`, qui calcule la même chose en stockant chaque étape dans une liste (`readlines`, `list(...)`).
4. Comparez les résultats et les pics de mémoire des deux versions avec `pic_memoire`. Les totaux doivent être égaux, et la version à base de listes doit utiliser une centaine de mégaoctets de plus.

In [ ]:
import random
import tracemalloc

with open("requetes.log", "w", encoding="utf8") as fh:
  for i in range(500_000):
    fh.write(f"{i};utilisateur{random.randrange(1_000)};{random.randrange(1_000)}\n")


def pic_memoire(fonction):
  """Exécute fonction() et renvoie son résultat et son pic de mémoire en Mo."""
  tracemalloc.start()
  resultat = fonction()
  _, pic = tracemalloc.get_traced_memory()
  tracemalloc.stop()
  return resultat, pic / 1e6

In [ ]:
# Votre code ici
from collections.abc import Iterable, Iterator


def lit_lignes(chemin: str) -> Iterator[str]: ...


def analyse(lignes: Iterable[str]) -> Iterator[tuple[str, int]]: ...


def lentes(
  requetes: Iterable[tuple[str, int]], seuil: int
) -> Iterator[tuple[str, int]]: ...


def total_lentes_generateurs(chemin: str) -> int: ...


def total_lentes_listes(chemin: str) -> int: ...

### Conseils supplémentaires

Chaque générateur parcourt ce qu'il reçoit avec `for` et produit avec `yield` ; `lit_lignes` ouvre le fichier dans un `with` et délègue à `fh` avec `yield from`. `ligne.rstrip("\n").split(";")` découpe une ligne en ses trois champs. `pic_memoire` prend une fonction sans argument, d'où le `lambda`, et renvoie le résultat de cette fonction et son pic de mémoire. Attendu : `True`, moins de 0,1 Mo pour les générateurs, une centaine de Mo pour les listes.

```python
def lit_lignes(chemin: str) -> Iterator[str]:
  with open(chemin, encoding="utf8") as fh:
    yield from ...


def analyse(lignes: Iterable[str]) -> Iterator[tuple[str, int]]:
  for ligne in lignes:
    _, utilisateur, duree = ...
    yield ...


def lentes(
  requetes: Iterable[tuple[str, int]], seuil: int
) -> Iterator[tuple[str, int]]:
  for utilisateur, duree in requetes:
    if ...:
      yield utilisateur, duree


def total_lentes_generateurs(chemin: str) -> int:
  return sum(duree for _, duree in ...)


def total_lentes_listes(chemin: str) -> int:
  with open(chemin, encoding="utf8") as fh:
    lignes = fh.readlines()
  requetes = list(...)
  return sum(duree for _, duree in list(...))


total_g, pic_g = pic_memoire(lambda: total_lentes_generateurs("requetes.log"))
total_l, pic_l = ...
print(total_g == total_l, pic_g, pic_l)
```

### Solution

In [ ]:
from collections.abc import Iterable, Iterator


def lit_lignes(chemin: str) -> Iterator[str]:
  with open(chemin, encoding="utf8") as fh:
    yield from fh


def analyse(lignes: Iterable[str]) -> Iterator[tuple[str, int]]:
  for ligne in lignes:
    _, utilisateur, duree = ligne.rstrip("\n").split(";")
    yield utilisateur, int(duree)


def lentes(
  requetes: Iterable[tuple[str, int]], seuil: int
) -> Iterator[tuple[str, int]]:
  for utilisateur, duree in requetes:
    if duree > seuil:
      yield utilisateur, duree


def total_lentes_generateurs(chemin: str) -> int:
  return sum(duree for _, duree in lentes(analyse(lit_lignes(chemin)), 900))


def total_lentes_listes(chemin: str) -> int:
  with open(chemin, encoding="utf8") as fh:
    lignes = fh.readlines()
  requetes = list(analyse(lignes))
  return sum(duree for _, duree in list(lentes(requetes, 900)))


total_g, pic_g = pic_memoire(lambda: total_lentes_generateurs("requetes.log"))
total_l, pic_l = pic_memoire(lambda: total_lentes_listes("requetes.log"))
print(total_g == total_l, f"générateurs : {pic_g:.1f} Mo, listes : {pic_l:.1f} Mo")

Les générateurs ne gardent en mémoire qu'une ligne à la fois, quelle que soit la taille du fichier : la version à base de listes stocke tout le fichier, puis tous les tuples.

## Libérer une ressource avec `close` (difficile)

Un générateur peut détenir une ressource (fichier, connexion…) pendant qu'il produit ses valeurs.

Écrivez un générateur `lignes_numerotees(chemin)` qui affiche `ouverture`, ouvre le fichier et produit ses lignes numérotées `(numéro, ligne)` avec `yield from`, puis affiche `fermeture` quand le fichier est refermé, y compris si l'appelant s'arrête avant la fin.

Lisez les trois premières lignes de `requetes.log`, le fichier créé dans l'exercice précédent, avec `next`, puis appelez `close()` sur le générateur : `fermeture` doit s'afficher immédiatement. Pourquoi ?

**Documentation utile**

- [`generator.close`](https://docs.python.org/fr/3/builtins/stdtypes.html#generator.close)
- [`GeneratorExit`](https://docs.python.org/fr/3/library/exceptions.html#GeneratorExit)

In [ ]:
# Votre code ici
from collections.abc import Iterator


def lignes_numerotees(chemin: str) -> Iterator[tuple[int, str]]: ...

### Conseils supplémentaires

`try`/`finally` entoure le `with` : le bloc `finally` s'exécute dès que le générateur se termine, qu'il ait tout lu ou qu'on l'ait interrompu. `enumerate(fh, start=1)` numérote les lignes à partir de 1, et `yield from` en produit chaque tuple. Attendu : `ouverture`, les tuples numérotés 1, 2 et 3, puis `fermeture`.

```python
def lignes_numerotees(chemin: str) -> Iterator[tuple[int, str]]:
  print("ouverture")
  try:
    with open(chemin, encoding="utf8") as fh:
      yield from ...
  finally:
    ...


lignes = lignes_numerotees("requetes.log")
for _ in range(3):
  print(...)
lignes.close()
```

### Solution

In [ ]:
from collections.abc import Iterator


def lignes_numerotees(chemin: str) -> Iterator[tuple[int, str]]:
  print("ouverture")
  try:
    with open(chemin, encoding="utf8") as fh:
      yield from enumerate(fh, start=1)
  finally:
    print("fermeture")


lignes = lignes_numerotees("requetes.log")
for _ in range(3):
  print(next(lignes))
lignes.close()

`close()` lève `GeneratorExit` au point de pause du générateur : l'exception remonte, le `with` referme le fichier, puis le bloc `finally` qui l'entoure affiche `fermeture`. Sans `close()`, le fichier ne serait refermé qu'à la destruction du générateur.

## Utilisation de la méthode `send` d'un générateur (très difficile)

Écrivez le générateur `interets(montant)`. Il produit le montant courant, reçoit par `send` un taux d'intérêt (0.05 pour 5 %), et produit le nouveau montant après application du taux.

Créez `montants = interets(100)` et démarrez-le avec `next`, qui doit donner 100. Envoyez-lui ensuite les taux 0.05, 0.08 et -0.02 : `send` doit donner environ 105.0, 113.4 et 111.132.

**Documentation utile**

- [`generator.send`](https://docs.python.org/fr/3/builtins/stdtypes.html#generator.send)

In [ ]:
# Votre code ici
from collections.abc import Generator


def interets(montant: float) -> Generator[float, float, None]: ...

### Conseils supplémentaires

`taux = yield montant` fait deux choses : produit `montant` et se met en pause, puis vaut, à la reprise, la valeur envoyée par `send`. `send` renvoie la valeur du `yield` suivant. Un générateur qui vient d'être créé n'est arrêté sur aucun `yield` : le `next` initial l'amène au premier, sinon `send` lève `TypeError`. Un taux de 5 % multiplie le montant par `1 + taux`.

```python
def interets(montant: float) -> Generator[float, float, None]:
  while True:
    taux = yield ...
    montant *= ...


montants = interets(100)
print(next(montants))
for taux in [0.05, 0.08, -0.02]:
  print(montants.send(taux))
```

### Solution

In [ ]:
from collections.abc import Generator


def interets(montant: float) -> Generator[float, float, None]:
  while True:
    taux = yield montant
    montant *= 1 + taux


montants = interets(100)
print(next(montants))  # Démarre le générateur : 100
for taux in [0.05, 0.08, -0.02]:
  print(montants.send(taux))  # Environ 105.0, 113.4, 111.132